# VNICT2026 GeoFormerDock — kiem tra + chuan bi du lieu tren Kaggle

Notebook nay lam 4 viec, theo thu tu tu re/nhanh den nang hon
(`docs/revision_plan_reviews.md`):

1. **Tier 1** — kiem tra 5 gia tri `--geo_ablation` moi them vao GeoFormerDock
   (`tools/verify_geo_ablation.py`): dung torch thuan, khong can `data/`.
2. **Tier 2** — smoke test pipeline training that (molgrid + torch + ignite) tren
   2 mau co san trong repo (`demo_inference/`), khong can tai `data/`.
3. **Tier 3** — chi lay 2 file `.types` can dung (`ref_uff_train0`/`test0`) roi
   tach validation split theo receptor (`tools/make_val_split.py`) — stream qua
   pipe, KHONG tai nguyen `paper_types.tar.gz` (6.5GB).
4. **Tier 4** — chi trich cau truc protein (`PDBbind2016.tar.gz`, 4.2GB nen) cho
   DUNG cac receptor duoc 2 file `.types` o Tier 3 tham chieu toi, bo qua phan
   con lai — tranh vua ton bang thong vua ton dia cho du lieu khong dung toi.

Chien luoc nay thay the cach tai nguyen khoi truoc day (da tung het dia tren
Kaggle, va rieng tren may lab truong con phat hien bang thong toi nguon du lieu
qua cham — vai KB/s).

**⚠️ TRUOC KHI CHAY:** vao **Settings (panel phai) → Internet → On**. Va **KHONG
bat GPU accelerator** cho toi Tier 4 — ca 4 tier o day deu chay CPU, bat GPU
som chi ton quota tuan cua ban ma khong dung den.

Neu bat ky Tier nao FAIL: dung lai, dan output vao chat cho Claude, DUNG chay tiep.


## 0. Kiem tra moi truong Kaggle da co san gi

In [ ]:
import sys, subprocess
print('Python:', sys.version)
for pkg in ['numpy', 'torch']:
    try:
        mod = __import__(pkg)
        print(f'{pkg}: {mod.__version__} (da co san)')
    except ImportError:
        print(f'{pkg}: CHUA CO — se can cai them')


## 1. Lay code moi nhat tu GitHub

Repo public: `https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git` — khong can dang nhap/lien ket tai khoan gi, chi can
Internet: On o buoc tren.

In [ ]:
import os
if os.path.isdir('/kaggle/working/VNICT2026_Docking_Paper'):
    print('/kaggle/working/VNICT2026_Docking_Paper da ton tai — pull thay vi clone lai')
    !cd /kaggle/working/VNICT2026_Docking_Paper && git pull
else:
    !cd /kaggle/working && git clone https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git


In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!git log --oneline -1
print()
print('>>> Dan dong commit tren vao chat de Claude xac nhan ban dang chay ban moi nhat.')


## 2. Tier 1 — kiem tra `--geo_ablation` (KHONG can GPU, KHONG can `data/`)

Chi dung `torch` (Kaggle da cai san). Kiem tra:
1. Ca 5 gia tri `--geo_ablation` deu construct + forward pass duoc.
2. `geo_ablation="none"` cho DUNG 1.594.573 tham so (bang so da bao cao trong bai).
3. Checkpoint cu (neu co trong repo) load duoc vao model "none" moi, khong
   missing/unexpected keys.

**Neu cell duoi FAIL: dung lai, dan toan bo output cho Claude, DUNG chay Tier 2/3.**

In [ ]:
!python3 tools/verify_geo_ablation.py


## 3. Tier 2 — smoke test pipeline training that (chi chay neu Tier 1 PASS)

Can cai them `molgrid` (thu vien tao luoi voxel, khong co san tren Kaggle) va
`pytorch-ignite`, `mlflow`. **`molgrid` yeu cau `numpy<2`** — cell duoi ep phien
ban numpy truoc, co the khien pip cai/ha cap vai goi khac Kaggle co san (binh
thuong, khong phai loi).

Dung `demo_inference/` co san trong repo (4 file `.gninatypes` + 1 file `.types`,
vai chuc KB) — KHONG tai `data/` 80GB. Chay 3 epoch tren 2 mau, mat vai giay.

In [ ]:
!pip install -q 'numpy<2' molgrid pytorch-ignite mlflow


### ⚠️ BAT BUOC: Restart kernel truoc khi chay tiep

`numpy` la mot C-extension — neu cell tren vua doi phien ban numpy,
`importlib.reload()` KHONG dang tin cay de nap lai dung ban moi (day la mot
gotcha quen thuoc cua Jupyter, khong rieng gi notebook nay). Cach chac chan
duy nhat: **Kernel menu → Restart Kernel** (khong chon "Restart & Run All" —
chi restart, roi tu chay tiep tu cell duoi, KHONG chay lai tu dau vi Tier 1
da xong roi).

Sau khi restart, kernel mat toan bo state (ke ca vi tri thu muc) — cell duoi
se tu `cd` lai vao repo va kiem tra numpy truoc khi chay smoke test.

In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
import numpy
print('numpy:', numpy.__version__)
assert numpy.__version__.startswith('1.'), (
    f'numpy={numpy.__version__} van >=2 sau khi cai va restart kernel — '
    'molgrid se import loi. Kiem tra lai cell pip install o tren.'
)
print('OK — numpy < 2, an toan de import molgrid.')


In [ ]:
!bash scripts/run_geoformerdock_ablations.sh smoketest


## 4. Tier 3 — chi lay 2 file `.types` can dung (KHONG tai nguyen `paper_types.tar.gz` 6.5GB)

`paper_types.tar.gz` gop chung `.types` cua RAT NHIEU bo split khac nhau cua
CrossDocked2020, nhung du an chi can dung 2 file: `ref_uff_train0.types` va
`ref_uff_test0.types`. Cell duoi **stream truc tiep qua pipe** (`curl | tar`),
KHONG BAO GIO ghi file `.tar.gz` 6.5GB xuong dia — chi ghi ra dung 2 file
`.types` can (vai chuc MB). Da tung thu cach nay tren may lab (bang thong rat
cham, ~8-10 KB/s) va bi timeout; Kaggle thuong co bang thong quoc te tot hon
nhieu nen cell nay CO THE nhanh hon dang ke — neu van cham/timeout, bao lai
Claude ngay, dung thu tiep.

In [ ]:
!mkdir -p data/types
!timeout 1800 curl -sL 'https://bits.csb.pitt.edu/files/crossdock2020/v1.0/paper_types.tar.gz' \
    | tar -xz -C data types/ref_uff_train0.types types/ref_uff_test0.types
!echo '--- ket qua ---'
!ls -la data/types/ 2>&1
!wc -l data/types/ref_uff_train0.types data/types/ref_uff_test0.types 2>&1


**Kiem tra truoc khi chay tiep:** cell tren phai in ra 2 file voi so dong
khop ky vong: `ref_uff_train0.types` = 62.335 dong, `ref_uff_test0.types` =
4.618 dong. Neu bao 'No such file or directory' hoac so dong = 0, DUNG lai —
bao Claude ngay, dung doan tiep vi cac buoc sau deu can 2 file nay.

In [ ]:
!python3 tools/make_val_split.py \
    --train data/types/ref_uff_train0.types \
    --out_train data/types/ref_uff_train0_split.types \
    --out_val data/types/ref_uff_val0.types \
    --val_frac 0.15 --seed 2026


**Kiem tra A0b (doc trong `docs/revision_plan_reviews.md`):** trong output cua
cell tren phai co dong `Giao receptor train/val (phai = 0)          : 0`.
Neu con so cuoi khac 0, DUNG lai va bao Claude — co loi ro ri du lieu.

In [ ]:
# Dem so mau y_aff > 0 trong tap val moi tach (A0b: can >= 500 de C-index
# tren val du on dinh de chon checkpoint).
def count_pos(path):
    n_lines = n_pos = 0
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            n_lines += 1
            if float(line.split()[1]) > 0:
                n_pos += 1
    return n_lines, n_pos

for split in ['train0_split', 'val0']:
    n, npos = count_pos(f'data/types/ref_uff_{split}.types')
    print(f'{split:14s}: {n:7d} dong, {npos:5d} mau y_aff>0 ({100*npos/n:.1f}%)')


## 5. Tier 4 — chi trich cau truc protein (PDBbind2016) cho DUNG cac receptor can dung

`PDBbind2016.tar.gz` (4.2GB nen) chua cau truc cua **toan bo** receptor trong
CrossDocked2020, nhung `ref_uff_train0`/`test0` chi tham chieu toi mot tap con.
Cell duoi:
1. Doc 2 file `.types` da co, lay danh sach **receptor rieng biet** duoc tham chieu.
2. Ghi danh sach do thanh file (moi dong 2 duong dan ung vien — vi CHUA BIET
   chac chan archive dung tien to `PDBbind2016/<ma>/` hay chi `<ma>/` — dua ca
   hai vao, `tar` se tu bo qua duong dan nao khong khop, khong loi).
3. Stream-extract CHI cac thu muc receptor do tu `PDBbind2016.tar.gz`, khong
   bao gio ghi file .tar.gz 4.2GB xuong dia.

In [ ]:
receptors = set()
for fname in ['data/types/ref_uff_train0.types', 'data/types/ref_uff_test0.types']:
    with open(fname) as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.split()
            # cot 3 (0-indexed) la receptor_path, dang '<ma>/<file>.gninatypes'
            receptor_path = parts[3]
            receptors.add(receptor_path.split('/')[0])

print(f'So receptor rieng biet can dung: {len(receptors)}')

with open('receptor_members.txt', 'w') as f:
    for r in sorted(receptors):
        f.write(f'PDBbind2016/{r}\n')  # ung vien 1: co tien to PDBbind2016/
        f.write(f'{r}\n')              # ung vien 2: khong co tien to

print(f'Da ghi receptor_members.txt ({len(receptors) * 2} dong, 2 ung vien/receptor)')


In [ ]:
# Chay curl|tar o NEN, tu kiem tra tien do moi 20s, TU DUNG khi:
#   (a) da co >=98% receptor can dung, HOAC
#   (b) 80s lien tuc khong tang them (nghia la da doc qua het cac muc can,
#       phan con lai cua archive khong con gi lien quan), HOAC
#   (c) qua 60 phut (gioi han an toan, tranh treo vo han).
# Giai quyet dung van de gap o Tier 3: tar doc tu pipe khong biet khi nao
# da du, se cu doc het luong neu khong ai chu dong dung no lai.
import os, signal, subprocess, time

os.makedirs('data', exist_ok=True)
# set -o pipefail: de returncode phan anh dung loi cua curl (vd 404) chu
# khong chi loi cua tar (mac dinh shell chi lay returncode lenh CUOI trong pipe).
cmd = "set -o pipefail; curl -sL 'https://bits.csb.pitt.edu/files/crossdock2020/PDBbind2016.tar.gz' | tar -xz -C data -T receptor_members.txt"
proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash', preexec_fn=os.setsid)

def coverage():
    found = sum(
        1 for r in receptors
        if os.path.isdir(f'data/PDBbind2016/{r}') or os.path.isdir(f'data/{r}')
    )
    return found, len(receptors)

last_found, stable_checks, start = -1, 0, time.time()
MAX_WAIT_S = 3600

while True:
    time.sleep(20)
    found, total = coverage()
    elapsed = time.time() - start
    print(f'[{elapsed:5.0f}s] receptor da co du lieu: {found}/{total} ({100*found/total:.1f}%)')

    if proc.poll() is not None:
        rc = proc.returncode
        status = 'THANH CONG (doc het archive)' if rc == 0 else f'LOI (returncode={rc}) — kiem tra lai URL/mang'
        print(f'curl|tar da tu ket thuc: {status}.')
        break

    stable_checks = stable_checks + 1 if found == last_found else 0
    last_found = found

    if found / total >= 0.98:
        print('>>> Da dat >=98% receptor — dung tien trinh, bo qua phan con lai cua archive.')
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        break
    if stable_checks >= 4:
        print(f'>>> Khong tang them sau {stable_checks * 20}s — co le da doc qua het cac muc '
              f'lien quan. Dung tien trinh (con {total - found} receptor CO THE khong co trong archive).')
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        break
    if elapsed >= MAX_WAIT_S:
        print('>>> Qua 60 phut — dung tien trinh. Bao Claude ket qua nay.')
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        break

time.sleep(2)
print()
print('--- ket qua ---')
!du -sh data/ 2>&1
!df -h /kaggle/working 2>&1 | tail -1


**Kiem tra doc lap — KHONG doan cau truc thu muc, kiem tra thang tren danh
sach receptor da biet** (2 ung vien duong dan co the tao ra do sau thu muc
khac nhau, nen dem thu muc theo do sau se sai; kiem tra nay dung cho ca 2
truong hop):

In [ ]:
import os
found_prefixed = found_flat = missing = 0
missing_examples = []
for r in receptors:
    if os.path.isdir(f'data/PDBbind2016/{r}'):
        found_prefixed += 1
    elif os.path.isdir(f'data/{r}'):
        found_flat += 1
    else:
        missing += 1
        if len(missing_examples) < 5:
            missing_examples.append(r)

total = len(receptors)
print(f'Tong receptor can: {total}')
print(f'  Tim thay dang data/PDBbind2016/<ma>/  : {found_prefixed}')
print(f'  Tim thay dang data/<ma>/ (khong tien to): {found_flat}')
print(f'  KHONG tim thay o ca 2 dang             : {missing} '
      f'({100*missing/total:.1f}%)')
if missing_examples:
    print(f'  Vi du receptor bi thieu: {missing_examples}')
if missing / total > 0.05:
    print()
    print('CANH BAO: thieu hon 5% receptor — DUNG lai, bao Claude ngay kem '
          'output nay va output cell truoc.')
else:
    print()
    print('OK — da co du du lieu cho gan het receptor can dung.')


**Loc bo cac dong tham chieu receptor khong co du lieu** — khong ro `molgrid`
se bo qua em hay bao loi khi gap file thieu (khong kiem chung duoc tu xa),
nen loc thang truoc cho chac, thay vi danh cuoc. Ghi de len chinh 3 file
dang dung cho training (`train0_split`, `val0`, `test0`), giu ban goc truoc
khi ghi de de doi chieu duoc so dong mat di.

In [ ]:
import os, shutil

missing_receptors = {
    r for r in receptors
    if not os.path.isdir(f'data/PDBbind2016/{r}') and not os.path.isdir(f'data/{r}')
}
print(f'So receptor se bi loc bo: {len(missing_receptors)}')

targets = [
    'data/types/ref_uff_train0_split.types',
    'data/types/ref_uff_val0.types',
    'data/types/ref_uff_test0.types',
]
for path in targets:
    backup = path + '.orig'
    if not os.path.exists(backup):
        shutil.copy(path, backup)
    with open(backup) as f:
        lines = [ln for ln in f if ln.strip()]
    kept = [ln for ln in lines if ln.split()[3].split('/')[0] not in missing_receptors]
    with open(path, 'w') as f:
        f.writelines(kept)
    print(f'{path}: {len(lines)} -> {len(kept)} dong (loai {len(lines) - len(kept)})')


## 6. Tom tat — dan phan nay vao chat cho Claude

Chay cell duoi roi copy toan bo output gui lai, kem ket qua Tier 1-4 o tren.

In [ ]:
import subprocess, os
print('=== TOM TAT DE GUI LAI ===')
print('git commit:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
print('numpy:', __import__('numpy').__version__)
try:
    print('torch:', __import__('torch').__version__)
except ImportError:
    print('torch: KHONG CO')
try:
    __import__('molgrid')
    print('molgrid: import OK')
except ImportError as e:
    print('molgrid: KHONG import duoc —', e)
for f in ['data/types/ref_uff_train0_split.types', 'data/types/ref_uff_val0.types']:
    print(f, '-> co san' if os.path.exists(f) else '-> THIEU')
# 'receptors' la bien da tinh o cell Tier 4 phia tren (cung kernel, con trong bo nho)
if 'receptors' in dir():
    n_ok = sum(
        1 for r in receptors
        if os.path.isdir(f'data/PDBbind2016/{r}') or os.path.isdir(f'data/{r}')
    )
    print(f'Receptor da co du lieu: {n_ok}/{len(receptors)}')
print(subprocess.run(['du', '-sh', 'data'], capture_output=True, text=True).stdout.strip())
